In [3]:
import os
import json
import numpy as np

FEATURE_ROOT = "/Users/canpasa/Desktop/NEW PAPER GTZAN/extracted_features_with_segments_new"

MASTER_CONFIG_DIR = os.path.join(
    FEATURE_ROOT,
    "fft1024_hop256_mel128_seg3"
)

MASTER_SPLIT_DIR = os.path.join(MASTER_CONFIG_DIR, "track_level_split")

print("MASTER_SPLIT_DIR:", MASTER_SPLIT_DIR)
print("Exists:", os.path.exists(MASTER_SPLIT_DIR))

MASTER_SPLIT_DIR: /Users/canpasa/Desktop/NEW PAPER GTZAN/extracted_features_with_segments_new/fft1024_hop256_mel128_seg3/track_level_split
Exists: False


In [4]:
import os
import json
import numpy as np
from sklearn.model_selection import train_test_split

FEATURE_DIR = "/Users/canpasa/Desktop/NEW PAPER GTZAN/extracted_features_with_segments_new/fft1024_hop256_mel128_seg3"
RANDOM_STATE = 42

X = np.load(os.path.join(FEATURE_DIR, "X.npy"))
y = np.load(os.path.join(FEATURE_DIR, "y.npy"))

with open(os.path.join(FEATURE_DIR, "segments.json"), "r", encoding="utf-8") as f:
    segments = json.load(f)

track_to_label = {}
for i, seg in enumerate(segments):
    track_path = seg["track_path"]
    label = int(y[i])

    if track_path not in track_to_label:
        track_to_label[track_path] = label
    else:
        if track_to_label[track_path] != label:
            raise ValueError(f"Label mismatch in track: {track_path}")

unique_tracks = np.array(list(track_to_label.keys()))
unique_track_labels = np.array([track_to_label[t] for t in unique_tracks])

train_tracks, temp_tracks, train_labels, temp_labels = train_test_split(
    unique_tracks,
    unique_track_labels,
    test_size=0.30,
    stratify=unique_track_labels,
    random_state=RANDOM_STATE
)

val_tracks, test_tracks, val_labels, test_labels = train_test_split(
    temp_tracks,
    temp_labels,
    test_size=0.50,
    stratify=temp_labels,
    random_state=RANDOM_STATE
)

train_track_set = set(train_tracks)
val_track_set = set(val_tracks)
test_track_set = set(test_tracks)

train_idx, val_idx, test_idx = [], [], []

for i, seg in enumerate(segments):
    track_path = seg["track_path"]
    if track_path in train_track_set:
        train_idx.append(i)
    elif track_path in val_track_set:
        val_idx.append(i)
    elif track_path in test_track_set:
        test_idx.append(i)

train_idx = np.array(train_idx)
val_idx = np.array(val_idx)
test_idx = np.array(test_idx)

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

split_dir = os.path.join(FEATURE_DIR, "track_level_split")
os.makedirs(split_dir, exist_ok=True)

np.save(os.path.join(split_dir, "X_train.npy"), X_train)
np.save(os.path.join(split_dir, "y_train.npy"), y_train)
np.save(os.path.join(split_dir, "X_val.npy"), X_val)
np.save(os.path.join(split_dir, "y_val.npy"), y_val)
np.save(os.path.join(split_dir, "X_test.npy"), X_test)
np.save(os.path.join(split_dir, "y_test.npy"), y_test)

np.save(os.path.join(split_dir, "train_idx.npy"), train_idx)
np.save(os.path.join(split_dir, "val_idx.npy"), val_idx)
np.save(os.path.join(split_dir, "test_idx.npy"), test_idx)

with open(os.path.join(split_dir, "train_tracks.json"), "w", encoding="utf-8") as f:
    json.dump(train_tracks.tolist(), f, indent=2, ensure_ascii=False)

with open(os.path.join(split_dir, "val_tracks.json"), "w", encoding="utf-8") as f:
    json.dump(val_tracks.tolist(), f, indent=2, ensure_ascii=False)

with open(os.path.join(split_dir, "test_tracks.json"), "w", encoding="utf-8") as f:
    json.dump(test_tracks.tolist(), f, indent=2, ensure_ascii=False)

print("Saved:", split_dir)
print(os.listdir(split_dir))

Saved: /Users/canpasa/Desktop/NEW PAPER GTZAN/extracted_features_with_segments_new/fft1024_hop256_mel128_seg3/track_level_split
['test_tracks.json', 'y_train.npy', 'train_tracks.json', 'val_tracks.json', 'test_idx.npy', 'train_idx.npy', 'y_test.npy', 'X_test.npy', 'val_idx.npy', 'y_val.npy', 'X_train.npy', 'X_val.npy']


In [5]:
import os

root = "/Users/canpasa/Desktop/NEW PAPER GTZAN/extracted_features_with_segments_new"

for d in sorted(os.listdir(root)):
    full = os.path.join(root, d)
    if os.path.isdir(full):
        print(d, "->", os.listdir(full))

fft1024_hop128_mel128_seg3 -> ['label_map.json', 'segments.json', 'bad_files.json', 'X.npy', 'y.npy', 'meta.json']
fft1024_hop256_mel128_seg3 -> ['track_level_split', 'label_map.json', 'segments.json', 'bad_files.json', 'X.npy', 'y.npy', 'meta.json']
fft1024_hop512_mel128_seg3 -> ['label_map.json', 'segments.json', 'bad_files.json', 'X.npy', 'y.npy', 'meta.json']
fft2048_hop128_mel128_seg3 -> ['label_map.json', 'segments.json', 'bad_files.json', 'X.npy', 'y.npy', 'meta.json']
fft2048_hop256_mel128_seg3 -> ['label_map.json', 'segments.json', 'bad_files.json', 'X.npy', 'y.npy', 'meta.json']
fft2048_hop512_mel128_seg3 -> ['label_map.json', 'segments.json', 'bad_files.json', 'X.npy', 'y.npy', 'meta.json']
fft512_hop128_mel128_seg3 -> ['label_map.json', 'segments.json', 'bad_files.json', 'X.npy', 'y.npy', 'meta.json']
fft512_hop256_mel128_seg3 -> ['label_map.json', 'segments.json', 'bad_files.json', 'X.npy', 'y.npy', 'meta.json']
fft512_hop512_mel128_seg3 -> ['label_map.json', 'segments.jso

In [6]:
import os
import json
import numpy as np

# =========================================================
# CONFIG
# =========================================================
FEATURE_ROOT = "/Users/canpasa/Desktop/NEW PAPER GTZAN/extracted_features_with_segments_new"
MASTER_CONFIG_NAME = "fft1024_hop256_mel128_seg3"

MASTER_CONFIG_DIR = os.path.join(FEATURE_ROOT, MASTER_CONFIG_NAME)
MASTER_SPLIT_DIR = os.path.join(MASTER_CONFIG_DIR, "track_level_split")

print("MASTER_SPLIT_DIR:", MASTER_SPLIT_DIR)
print("Exists:", os.path.exists(MASTER_SPLIT_DIR))
print("Master split files:", os.listdir(MASTER_SPLIT_DIR))

# =========================================================
# LOAD MASTER SPLIT
# =========================================================
with open(os.path.join(MASTER_SPLIT_DIR, "train_tracks.json"), "r", encoding="utf-8") as f:
    train_tracks = json.load(f)

with open(os.path.join(MASTER_SPLIT_DIR, "val_tracks.json"), "r", encoding="utf-8") as f:
    val_tracks = json.load(f)

with open(os.path.join(MASTER_SPLIT_DIR, "test_tracks.json"), "r", encoding="utf-8") as f:
    test_tracks = json.load(f)

train_track_set = set(train_tracks)
val_track_set = set(val_tracks)
test_track_set = set(test_tracks)

print("\n[INFO] Master split loaded.")
print("Train tracks:", len(train_track_set))
print("Val tracks  :", len(val_track_set))
print("Test tracks :", len(test_track_set))

# =========================================================
# HELPER
# =========================================================
def apply_master_split_to_feature_dir(feature_dir):
    config_name = os.path.basename(feature_dir)

    # master klasörü atla istersen
    if config_name == MASTER_CONFIG_NAME:
        print(f"\n[SKIP] Master config already has split: {config_name}")
        return

    print("\n" + "=" * 70)
    print(f"[PROCESSING] {config_name}")
    print("=" * 70)

    x_path = os.path.join(feature_dir, "X.npy")
    y_path = os.path.join(feature_dir, "y.npy")
    segments_path = os.path.join(feature_dir, "segments.json")

    if not (os.path.exists(x_path) and os.path.exists(y_path) and os.path.exists(segments_path)):
        print("[SKIP] Missing required files.")
        return

    X = np.load(x_path)
    y = np.load(y_path)

    with open(segments_path, "r", encoding="utf-8") as f:
        segments = json.load(f)

    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("segments:", len(segments))

    if not (len(X) == len(y) == len(segments)):
        print("[SKIP] Length mismatch between X, y, and segments.")
        return

    train_idx, val_idx, test_idx = [], [], []

    for i, seg in enumerate(segments):
        track_path = seg["track_path"]

        if track_path in train_track_set:
            train_idx.append(i)
        elif track_path in val_track_set:
            val_idx.append(i)
        elif track_path in test_track_set:
            test_idx.append(i)
        else:
            raise ValueError(f"Track not found in master split: {track_path}")

    train_idx = np.array(train_idx, dtype=np.int64)
    val_idx = np.array(val_idx, dtype=np.int64)
    test_idx = np.array(test_idx, dtype=np.int64)

    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    print("\n[SPLIT SHAPES]")
    print("X_train:", X_train.shape, "y_train:", y_train.shape)
    print("X_val  :", X_val.shape, "y_val  :", y_val.shape)
    print("X_test :", X_test.shape, "y_test :", y_test.shape)

    split_dir = os.path.join(feature_dir, "track_level_split")
    os.makedirs(split_dir, exist_ok=True)

    np.save(os.path.join(split_dir, "X_train.npy"), X_train)
    np.save(os.path.join(split_dir, "y_train.npy"), y_train)
    np.save(os.path.join(split_dir, "X_val.npy"), X_val)
    np.save(os.path.join(split_dir, "y_val.npy"), y_val)
    np.save(os.path.join(split_dir, "X_test.npy"), X_test)
    np.save(os.path.join(split_dir, "y_test.npy"), y_test)

    np.save(os.path.join(split_dir, "train_idx.npy"), train_idx)
    np.save(os.path.join(split_dir, "val_idx.npy"), val_idx)
    np.save(os.path.join(split_dir, "test_idx.npy"), test_idx)

    with open(os.path.join(split_dir, "train_tracks.json"), "w", encoding="utf-8") as f:
        json.dump(sorted(list(train_track_set)), f, indent=2, ensure_ascii=False)

    with open(os.path.join(split_dir, "val_tracks.json"), "w", encoding="utf-8") as f:
        json.dump(sorted(list(val_track_set)), f, indent=2, ensure_ascii=False)

    with open(os.path.join(split_dir, "test_tracks.json"), "w", encoding="utf-8") as f:
        json.dump(sorted(list(test_track_set)), f, indent=2, ensure_ascii=False)

    print(f"[SAVED] {split_dir}")

# =========================================================
# RUN
# =========================================================
feature_dirs = sorted([
    os.path.join(FEATURE_ROOT, d)
    for d in os.listdir(FEATURE_ROOT)
    if os.path.isdir(os.path.join(FEATURE_ROOT, d))
])

print("\n[INFO] Found feature folders:", len(feature_dirs))

for feature_dir in feature_dirs:
    apply_master_split_to_feature_dir(feature_dir)

print("\n[INFO] Done.")

MASTER_SPLIT_DIR: /Users/canpasa/Desktop/NEW PAPER GTZAN/extracted_features_with_segments_new/fft1024_hop256_mel128_seg3/track_level_split
Exists: True
Master split files: ['test_tracks.json', 'y_train.npy', 'train_tracks.json', 'val_tracks.json', 'test_idx.npy', 'train_idx.npy', 'y_test.npy', 'X_test.npy', 'val_idx.npy', 'y_val.npy', 'X_train.npy', 'X_val.npy']

[INFO] Master split loaded.
Train tracks: 699
Val tracks  : 150
Test tracks : 150

[INFO] Found feature folders: 9

[PROCESSING] fft1024_hop128_mel128_seg3
X shape: (9990, 128, 517, 1)
y shape: (9990,)
segments: 9990

[SPLIT SHAPES]
X_train: (6990, 128, 517, 1) y_train: (6990,)
X_val  : (1500, 128, 517, 1) y_val  : (1500,)
X_test : (1500, 128, 517, 1) y_test : (1500,)
[SAVED] /Users/canpasa/Desktop/NEW PAPER GTZAN/extracted_features_with_segments_new/fft1024_hop128_mel128_seg3/track_level_split

[SKIP] Master config already has split: fft1024_hop256_mel128_seg3

[PROCESSING] fft1024_hop512_mel128_seg3
X shape: (9990, 128, 130,

In [7]:
import os
import json

FEATURE_DIR = "/Users/canpasa/Desktop/NEW PAPER GTZAN/extracted_features_with_segments_new/fft1024_hop256_mel128_seg3/track_level_split"

with open(os.path.join(FEATURE_DIR, "train_tracks.json")) as f:
    train_tracks = set(json.load(f))

with open(os.path.join(FEATURE_DIR, "val_tracks.json")) as f:
    val_tracks = set(json.load(f))

with open(os.path.join(FEATURE_DIR, "test_tracks.json")) as f:
    test_tracks = set(json.load(f))

print("train ∩ val:", train_tracks.intersection(val_tracks))
print("train ∩ test:", train_tracks.intersection(test_tracks))
print("val ∩ test:", val_tracks.intersection(test_tracks))

train ∩ val: set()
train ∩ test: set()
val ∩ test: set()


In [8]:
import numpy as np
import json
import os

FEATURE_DIR = "/Users/canpasa/Desktop/NEW PAPER GTZAN/extracted_features_with_segments_new/fft1024_hop256_mel128_seg3"

with open(os.path.join(FEATURE_DIR, "segments.json")) as f:
    segments = json.load(f)

split_dir = os.path.join(FEATURE_DIR, "track_level_split")

train_idx = np.load(os.path.join(split_dir, "train_idx.npy"))
val_idx = np.load(os.path.join(split_dir, "val_idx.npy"))
test_idx = np.load(os.path.join(split_dir, "test_idx.npy"))

train_tracks = set(segments[i]["track_path"] for i in train_idx)
val_tracks = set(segments[i]["track_path"] for i in val_idx)
test_tracks = set(segments[i]["track_path"] for i in test_idx)

print("train-val overlap:", train_tracks & val_tracks)
print("train-test overlap:", train_tracks & test_tracks)
print("val-test overlap:", val_tracks & test_tracks)

train-val overlap: set()
train-test overlap: set()
val-test overlap: set()


In [9]:
import os
import json
import numpy as np

FEATURE_ROOT = "/Users/canpasa/Desktop/NEW PAPER GTZAN/extracted_features_with_segments_new"

def check_leakage_for_config(feature_dir):
    config_name = os.path.basename(feature_dir)
    split_dir = os.path.join(feature_dir, "track_level_split")
    segments_path = os.path.join(feature_dir, "segments.json")

    print("\n" + "=" * 70)
    print(f"[CHECKING] {config_name}")
    print("=" * 70)

    required = [
        os.path.join(split_dir, "train_tracks.json"),
        os.path.join(split_dir, "val_tracks.json"),
        os.path.join(split_dir, "test_tracks.json"),
        os.path.join(split_dir, "train_idx.npy"),
        os.path.join(split_dir, "val_idx.npy"),
        os.path.join(split_dir, "test_idx.npy"),
        segments_path
    ]

    for p in required:
        if not os.path.exists(p):
            print("[MISSING]", p)
            return

    with open(os.path.join(split_dir, "train_tracks.json"), "r", encoding="utf-8") as f:
        train_tracks = set(json.load(f))

    with open(os.path.join(split_dir, "val_tracks.json"), "r", encoding="utf-8") as f:
        val_tracks = set(json.load(f))

    with open(os.path.join(split_dir, "test_tracks.json"), "r", encoding="utf-8") as f:
        test_tracks = set(json.load(f))

    train_idx = np.load(os.path.join(split_dir, "train_idx.npy"))
    val_idx = np.load(os.path.join(split_dir, "val_idx.npy"))
    test_idx = np.load(os.path.join(split_dir, "test_idx.npy"))

    with open(segments_path, "r", encoding="utf-8") as f:
        segments = json.load(f)

    # 1) JSON track list overlap
    tv = train_tracks & val_tracks
    tt = train_tracks & test_tracks
    vt = val_tracks & test_tracks

    print("[TRACK JSON OVERLAP]")
    print("train ∩ val :", len(tv))
    print("train ∩ test:", len(tt))
    print("val ∩ test  :", len(vt))

    # 2) Indexlerden track üretip tekrar kontrol
    train_tracks_from_idx = set(segments[i]["track_path"] for i in train_idx)
    val_tracks_from_idx = set(segments[i]["track_path"] for i in val_idx)
    test_tracks_from_idx = set(segments[i]["track_path"] for i in test_idx)

    tv_idx = train_tracks_from_idx & val_tracks_from_idx
    tt_idx = train_tracks_from_idx & test_tracks_from_idx
    vt_idx = val_tracks_from_idx & test_tracks_from_idx

    print("\n[TRACK OVERLAP FROM INDICES]")
    print("train ∩ val :", len(tv_idx))
    print("train ∩ test:", len(tt_idx))
    print("val ∩ test  :", len(vt_idx))

    # 3) Track count sanity check
    print("\n[TRACK COUNTS]")
    print("train:", len(train_tracks_from_idx))
    print("val  :", len(val_tracks_from_idx))
    print("test :", len(test_tracks_from_idx))
    print("total unique:", len(train_tracks_from_idx | val_tracks_from_idx | test_tracks_from_idx))

    # 4) Sample count sanity check
    print("\n[SAMPLE COUNTS]")
    print("train_idx:", len(train_idx))
    print("val_idx  :", len(val_idx))
    print("test_idx :", len(test_idx))
    print("total    :", len(train_idx) + len(val_idx) + len(test_idx))
    print("segments :", len(segments))

    # Final yorum
    if len(tv) == len(tt) == len(vt) == len(tv_idx) == len(tt_idx) == len(vt_idx) == 0:
        print("\n[RESULT] No track leakage detected.")
    else:
        print("\n[RESULT] Leakage detected. Intersections are not empty.")

# tüm klasörler için
for d in sorted(os.listdir(FEATURE_ROOT)):
    feature_dir = os.path.join(FEATURE_ROOT, d)
    if os.path.isdir(feature_dir):
        check_leakage_for_config(feature_dir)


[CHECKING] fft1024_hop128_mel128_seg3
[TRACK JSON OVERLAP]
train ∩ val : 0
train ∩ test: 0
val ∩ test  : 0

[TRACK OVERLAP FROM INDICES]
train ∩ val : 0
train ∩ test: 0
val ∩ test  : 0

[TRACK COUNTS]
train: 699
val  : 150
test : 150
total unique: 999

[SAMPLE COUNTS]
train_idx: 6990
val_idx  : 1500
test_idx : 1500
total    : 9990
segments : 9990

[RESULT] No track leakage detected.

[CHECKING] fft1024_hop256_mel128_seg3
[TRACK JSON OVERLAP]
train ∩ val : 0
train ∩ test: 0
val ∩ test  : 0

[TRACK OVERLAP FROM INDICES]
train ∩ val : 0
train ∩ test: 0
val ∩ test  : 0

[TRACK COUNTS]
train: 699
val  : 150
test : 150
total unique: 999

[SAMPLE COUNTS]
train_idx: 6990
val_idx  : 1500
test_idx : 1500
total    : 9990
segments : 9990

[RESULT] No track leakage detected.

[CHECKING] fft1024_hop512_mel128_seg3
[TRACK JSON OVERLAP]
train ∩ val : 0
train ∩ test: 0
val ∩ test  : 0

[TRACK OVERLAP FROM INDICES]
train ∩ val : 0
train ∩ test: 0
val ∩ test  : 0

[TRACK COUNTS]
train: 699
val  : 150
te